<a href="https://colab.research.google.com/github/AbhijithDuggaraju/ProjectSelfcare/blob/main/ProjectSelfCare.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import os
import joblib
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.dummy import DummyClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    StackingClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score,
)
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    roc_auc_score,
)
from sklearn.calibration import CalibratedClassifierCV



HAIR_CONCERN_FALLBACK = {
    "oily_scalp": "dandruff",
    "frizz": "breakage",
    "early_hair_loss": "thinning",
    "gray_coverage": "thinning",
    "damage_from_styling": "breakage",
    "gray_hair": "significant_thinning",
    "scalp_health": "dandruff",
}

TREATMENT_CLASSES = ["home", "mixed", "medical"]

ORDINAL_MAPS = {
    "severity":          ["mild", "moderate", "severe"],
    "hair_damage_level": ["low", "moderate", "high"],
    "stress_level":      ["low", "moderate", "high"],
    "sleep_time":        ["greater_than_6hrs", "less_than_6hrs"],
    "age_category":      ["teen", "young_adult", "adult", "senior"],
}


REQUIRED_KEYS = [
    "age", "age_category", "gender", "skin_type",
    "concern_type", "severity", "sleep_time",
    "stress_level", "hair_concern", "hair_damage_level",
]



def validate(value, allowed, default):
    value = str(value).lower().strip()
    return value if value in allowed else default


def categorize_age(age: int) -> str:
    if 13 <= age <= 19:   return "teen"
    elif 20 <= age <= 34: return "young_adult"
    elif 35 <= age <= 49: return "adult"
    else:                 return "senior"


def score_treatment(row: pd.Series) -> int:
    score = 0
    sev     = row["severity"]
    dmg     = row["hair_damage_level"]
    stress  = row["stress_level"]
    sleep   = row["sleep_time"]
    age_cat = row["age_category"]
    skin    = row["skin_type"]

    if sev == "severe" and dmg == "high":
        score += 6
    elif sev == "severe":
        score += 4
    elif sev == "moderate" and dmg == "high":
        score += 4
    elif sev == "moderate":
        score += 2
    elif dmg == "high":
        score += 2

    if stress == "high" and sleep == "less_than_6hrs":
        score += 3
    elif stress == "high":
        score += 2
    elif stress == "moderate":
        score += 1

    if sleep == "less_than_6hrs":
        score += 1

    age_bonus = {"teen": 0, "young_adult": 0, "adult": 1, "senior": 2}
    score += age_bonus.get(age_cat, 0)

    if skin == "sensitive":
        score += 1

    concern = row.get("concern_type", "")
    if concern in ["deep_wrinkles", "aging", "dark_spots"]:
        score += 1
    hair = row.get("hair_concern", "")
    if hair in ["significant_thinning", "early_hair_loss"]:
        score += 1

    return score


BOUNDARY_ZONES = {
    "home_mixed":    (3, 5),
    "mixed_medical": (7, 9),
}


def assign_treatment(row: pd.Series, add_noise: bool = False) -> str:
    score = score_treatment(row)

    if add_noise:
        lo_home, hi_home   = BOUNDARY_ZONES["home_mixed"]
        lo_med,  hi_med    = BOUNDARY_ZONES["mixed_medical"]
        if lo_home <= score <= hi_home:
            return np.random.choice(["mixed", "home"], p=[0.60, 0.40])
        if lo_med  <= score <= hi_med:
            return np.random.choice(["mixed", "medical"], p=[0.60, 0.40])

    if score >= 8:   return "medical"
    elif score >= 4: return "mixed"
    else:            return "home"




def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()

    sev_map  = {"mild": 0, "moderate": 1, "severe": 2}
    dmg_map  = {"low": 0, "moderate": 1, "high": 2}
    str_map  = {"low": 0, "moderate": 1, "high": 2}
    slp_map  = {"greater_than_6hrs": 0, "less_than_6hrs": 1}
    age_map  = {"teen": 0, "young_adult": 1, "adult": 2, "senior": 3}

    d["sev_num"]  = d["severity"].map(sev_map).fillna(0)
    d["dmg_num"]  = d["hair_damage_level"].map(dmg_map).fillna(0)
    d["str_num"]  = d["stress_level"].map(str_map).fillna(0)
    d["slp_num"]  = d["sleep_time"].map(slp_map).fillna(0)
    d["age_num"]  = d["age_category"].map(age_map).fillna(0)


    d["sev_x_dmg"]       = d["sev_num"] * d["dmg_num"]
    d["stress_x_sleep"]  = d["str_num"] * d["slp_num"]
    d["age_x_sev"]       = d["age_num"] * d["sev_num"]
    d["age_x_stress"]    = d["age_num"] * d["str_num"]


    d["lifestyle_risk"]  = d["str_num"] + d["slp_num"]
    d["physical_risk"]   = d["sev_num"] + d["dmg_num"]


    d["is_senior"]        = (d["age_category"] == "senior").astype(int)
    d["is_sensitive_skin"]= (d["skin_type"] == "sensitive").astype(int)
    d["severe_flag"]      = (d["severity"] == "severe").astype(int)
    d["high_damage_flag"] = (d["hair_damage_level"] == "high").astype(int)
    d["poor_sleep_flag"]  = (d["sleep_time"] == "less_than_6hrs").astype(int)
    d["high_stress_flag"] = (d["stress_level"] == "high").astype(int)


    medical_skin = {"deep_wrinkles", "aging", "dark_spots"}
    medical_hair = {"significant_thinning", "early_hair_loss"}
    d["medical_concern_flag"] = (
        d["concern_type"].isin(medical_skin) |
        d["hair_concern"].isin(medical_hair)
    ).astype(int)

    return d




def build_stacked_model() -> StackingClassifier:
    rf = RandomForestClassifier(
        n_estimators=300,
        max_depth=8,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    )
    gb = GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        random_state=42,
    )
    rf_shallow = RandomForestClassifier(
        n_estimators=100,
        max_depth=4,
        random_state=7,
        n_jobs=-1,
    )


    meta = LogisticRegression(
        C=1.0,
        max_iter=1000,
        random_state=42,
        class_weight="balanced",
    )

    stacked = StackingClassifier(
        estimators=[
            ("rf",         rf),
            ("gb",         gb),
            ("rf_shallow", rf_shallow),
        ],
        final_estimator=meta,
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
        passthrough=True,
        n_jobs=-1,
    )
    return stacked




class CollaborativeFilter:
    def __init__(self, k: int = 10):
        self.k = k
        self.nn = NearestNeighbors(n_neighbors=k + 1, metric="euclidean")
        self.X_ref = None
        self.y_ref = None
        self.label_encoder = LabelEncoder()

    def fit(self, X: np.ndarray, y: np.ndarray):
        self.X_ref = X
        self.y_ref = self.label_encoder.fit_transform(y)
        self.nn.fit(X)

    def neighbor_prior(self, x: np.ndarray) -> np.ndarray:
        distances, indices = self.nn.kneighbors(x.reshape(1, -1))
        neighbor_idx = indices[0][1:]
        neighbor_labels = self.y_ref[neighbor_idx]

        dists = distances[0][1:] + 1e-6
        weights = 1.0 / dists
        weights /= weights.sum()

        n_classes = len(self.label_encoder.classes_)
        prior = np.zeros(n_classes)
        for label, w in zip(neighbor_labels, weights):
            prior[label] += w
        return prior




class SelfCareRecommender:

    def __init__(self, n_neighbors: int = 10, cf_weight: float = 0.25):
        self.cf_weight = cf_weight
        self.n_neighbors = n_neighbors


        self.categorical_cols = [
            "skin_type",
            "concern_type",
            "hair_concern",
            "gender",
        ]
        self.numeric_cols = ["age"]
        self.engineered_numeric = [
            "sev_num", "dmg_num", "str_num", "slp_num", "age_num",
            "sev_x_dmg", "stress_x_sleep", "age_x_sev", "age_x_stress",

            "lifestyle_risk", "physical_risk",
            "is_senior", "is_sensitive_skin", "severe_flag",
            "high_damage_flag", "poor_sleep_flag", "high_stress_flag",
            "medical_concern_flag",
        ]

        self.label_encoders: dict = {}
        self.scaler = StandardScaler()
        self.feature_columns = None

        self.base_model = build_stacked_model()
        self.calibrated_model = None
        self.cf = CollaborativeFilter(k=n_neighbors)

        self.treatment_encoder = LabelEncoder()
        self.feature_importances_ = None
        self.X_train_scaled = None



    def _encode_categoricals(self, df: pd.DataFrame, fit: bool) -> pd.DataFrame:
        for col in self.categorical_cols:
            if col not in df.columns:
                if fit:
                    df[col] = "unknown"
                else:
                    df[col] = self.label_encoders[col].classes_[0]

            if fit:
                le = LabelEncoder()
                df[col] = le.fit_transform(df[col].astype(str))
                self.label_encoders[col] = le
            else:
                le = self.label_encoders[col]
                known = set(le.classes_)
                df[col] = df[col].astype(str).apply(
                    lambda v: le.transform([v])[0] if v in known
                    else le.transform([le.classes_[0]])[0]
                )
        return df

    def _prepare(self, df: pd.DataFrame, fit: bool) -> pd.DataFrame:
        df = engineer_features(df)
        df = self._encode_categoricals(df, fit=fit)

        all_numeric = self.numeric_cols + self.engineered_numeric
        for col in all_numeric:
            if col not in df.columns:
                df[col] = 0

        all_features = self.categorical_cols + all_numeric

        if fit:
            self.scaler.fit(df[all_features])
            self.feature_columns = all_features

        return pd.DataFrame(
            self.scaler.transform(df[all_features]),
            columns=self.feature_columns,
        )

    def fit_preprocess(self, data: pd.DataFrame) -> pd.DataFrame:
        return self._prepare(data.copy(), fit=True)

    def transform_preprocess(self, data: pd.DataFrame) -> pd.DataFrame:
        return self._prepare(data.copy(), fit=False)



    def train_model(self, X: pd.DataFrame, y: pd.Series) -> dict:
        y_enc = self.treatment_encoder.fit_transform(y)


        print("Running 5-fold cross-validation...")
        cv_scores = cross_val_score(
            self.base_model, X, y_enc,
            cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
            scoring="accuracy",
            n_jobs=-1,
        )
        print(f"  CV Accuracy: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")


        X_tr, X_te, y_tr, y_te = train_test_split(
            X, y_enc, test_size=0.2,
            stratify=y_enc, random_state=42,
        )


        print("Fitting stacked ensemble...")
        self.base_model.fit(X_tr, y_tr)


        preds = self.base_model.predict(X_te)
        acc = accuracy_score(y_te, preds)
        proba = self.base_model.predict_proba(X_te)
        try:
            auc = roc_auc_score(y_te, proba, multi_class="ovr")
        except Exception:
            auc = None

        print(f"\n  Hold-out Accuracy : {acc:.3f}")
        if auc:
            print(f"  ROC-AUC (OvR)     : {auc:.3f}")


        dummy_maj = DummyClassifier(strategy="most_frequent")
        dummy_maj.fit(X_tr, y_tr)
        baseline_acc = accuracy_score(y_te, dummy_maj.predict(X_te))

        dummy_strat = DummyClassifier(strategy="stratified", random_state=42)
        dummy_strat.fit(X_tr, y_tr)
        strat_acc = accuracy_score(y_te, dummy_strat.predict(X_te))

        print(f"\n  Majority-class baseline  : {baseline_acc:.3f}")
        print(f"  Stratified baseline      : {strat_acc:.3f}")
        print(f"  Model improvement vs maj : +{acc - baseline_acc:.3f}")

        print("\nClassification Report (hold-out):")
        print(classification_report(
            y_te, preds,
            target_names=self.treatment_encoder.classes_,
        ))


        print("Calibrating stacked model on hold-out set...")
        self.calibrated_model = CalibratedClassifierCV(
            self.base_model,
            method="isotonic",
            cv="prefit",
        )
        self.calibrated_model.fit(X_te, y_te)

        print("Fitting collaborative filter...")
        self.cf.fit(X.values, y.values)
        self.X_train_scaled = X.values


        rf_estimator = self.base_model.named_estimators_["rf"]
        self.feature_importances_ = pd.Series(
            rf_estimator.feature_importances_,
            index=self.feature_columns,
        ).sort_values(ascending=False)

        return {
            "cv_mean": cv_scores.mean(),
            "cv_std": cv_scores.std(),
            "holdout_accuracy": acc,
            "roc_auc": auc,
            "majority_baseline": baseline_acc,
            "improvement_vs_baseline": acc - baseline_acc,
        }



    def get_recommendations(self, user_data: dict) -> dict:

        missing = [k for k in REQUIRED_KEYS if k not in user_data]
        if missing:
            raise ValueError(
                f"Missing required fields: {missing}. "
                f"All required: {REQUIRED_KEYS}"
            )

        X = self.transform_preprocess(pd.DataFrame([user_data]))
        x_vec = X.values[0]

        model_proba    = self.calibrated_model.predict_proba(X)[0]
        neighbor_proba = self.cf.neighbor_prior(x_vec)

        blended = (
            (1 - self.cf_weight) * model_proba +
            self.cf_weight       * neighbor_proba
        )
        blended /= blended.sum()

        max_p = blended.max()
        temperature = 1.5 if max_p < 0.75 else 1.0
        blended_logits = np.log(blended + 1e-9)
        blended = np.exp(blended_logits / temperature)
        blended /= blended.sum()

        predicted_class_idx = int(np.argmax(blended))
        predicted_class = self.treatment_encoder.classes_[predicted_class_idx]
        confidence = float(blended[predicted_class_idx])

        if confidence >= 0.80:
            uncertainty_tier = "high_confidence"
        elif confidence >= 0.55:
            uncertainty_tier = "moderate_confidence"
        else:
            uncertainty_tier = "low_confidence_consider_consultation"

        explanation = self._explain(x_vec, predicted_class, blended)

        skin_recs = self.format_skin_recommendations(
            probs=blended,
            user_data=user_data,
            treatment_type=predicted_class,
            confidence=confidence,
        )
        hair_recs = self.format_hair_recommendations(
            probs=blended,
            user_data=user_data,
            treatment_type=predicted_class,
            confidence=confidence,
        )

        return {
            "skin_care": skin_recs,
            "hair_care": hair_recs,
            "model_confidence": round(confidence, 3),
            "uncertainty_tier": uncertainty_tier,
            "model_probabilities": {
                cls: round(float(p), 3)
                for cls, p in zip(self.treatment_encoder.classes_, blended)
            },
            "explanation": explanation,

            "disclaimer": (
                "IMPORTANT: This is a prototype ML system trained on synthetic data. "
                "It is NOT a substitute for professional medical advice. "
                "Always consult a qualified dermatologist or trichologist "
                "before starting any medical or prescription treatment."
            ),
        }

    def _explain(self, x_vec: np.ndarray, predicted: str, probs: np.ndarray) -> dict:
        if self.feature_importances_ is None:
            return {}

        raw = self.feature_importances_.values * np.abs(x_vec)
        contrib = pd.Series(raw, index=self.feature_columns)
        top = contrib.nlargest(5)

        readable = {
            "sev_x_dmg":            "Severity × Hair Damage (combined risk)",
            "stress_x_sleep":       "Stress × Sleep Deprivation",
            "age_x_sev":            "Age × Severity",
            "age_x_stress":         "Age × Stress",
            "severe_flag":          "Severe condition",
            "high_damage_flag":     "High hair damage",
            "is_senior":            "Senior age group",
            "sev_num":              "Condition severity",
            "dmg_num":              "Hair damage level",
            "str_num":              "Stress level",
            "physical_risk":        "Physical risk (severity + damage)",
            "lifestyle_risk":       "Lifestyle risk (stress + sleep)",
            "medical_concern_flag": "Medical-grade concern type",
        }

        return {
            "predicted_treatment": predicted,
            "confidence_pct": f"{float(probs.max())*100:.1f}%",
            "top_drivers": [
                readable.get(f, f) for f in top.index.tolist()
            ],
        }

    def print_feature_importances(self, top_n: int = 10):
        if self.feature_importances_ is None:
            print("Model not trained yet.")
            return
        print(f"\n{'─'*45}")
        print("TOP FEATURE IMPORTANCES (Random Forest)")
        print(f"{'─'*45}")
        for feat, imp in self.feature_importances_.head(top_n).items():
            bar = "█" * int(imp * 200)
            print(f"  {feat:<28} {imp:.4f}  {bar}")



    def format_skin_recommendations(self, probs, user_data, treatment_type, confidence):
        sleep_time = user_data.get("sleep_time", "greater_than_6hrs")
        gender = user_data.get("gender", "female")

        severity_based_recommendations = {
            "teen": {
                "acne": {
                    "mild": [{"name": "Natural Acne Home Remedy Kit", "category": "home_remedy",
                        "confidence": round(confidence, 2), "cost": 12.99, "duration": "4 weeks",
                        "steps": (["Honey and Cinnamon Mask (2x weekly)", "Tea Tree Oil Spot Treatment",
                            "Apple Cider Vinegar Toner", "Aloe Vera Gel Moisturizer"]
                            if gender == "female" else
                            ["Honey Mask (2x weekly)", "Tea Tree Oil Spot Treatment",
                            "Witch Hazel Toner", "Light Gel Moisturizer"])
                            if sleep_time == "greater_than_6hrs" else
                            (["Oatmeal Soothing Cleanser", "Tea Tree Oil Spot Treatment",
                            "Chamomile Toner", "Light Aloe Moisturizer"]
                            if gender == "female" else
                            ["Oatmeal Cleanser", "Mild Tea Tree Treatment",
                            "Simple Toner", "Basic Moisturizer"])}],
                    "moderate": [{"name": "DIY Acne Management System", "category": "home_remedy",
                        "confidence": round(confidence, 2), "cost": 19.99, "duration": "6 weeks",
                        "steps": (["Green Tea Extract Cleanser", "Witch Hazel Toner",
                            "Tea Tree Oil Spot Treatment", "Yogurt and Turmeric Mask (weekly)",
                            "Aloe Vera Gel Moisturizer"]
                            if gender == "female" else
                            ["Green Tea Cleanser", "Witch Hazel Toner", "Tea Tree Oil Treatment",
                            "Clay Mask (weekly)", "Light Gel Moisturizer"])
                            if sleep_time == "greater_than_6hrs" else
                            (["Calendula Infused Cleanser", "Witch Hazel Toner",
                            "Diluted Tea Tree Oil Treatment", "Oatmeal Mask (weekly)",
                            "Light Hydrating Gel"]
                            if gender == "female" else
                            ["Simple Calendula Cleanser", "Basic Toner",
                            "Light Tea Tree Treatment", "Gentle Oatmeal Mask",
                            "Basic Hydrating Gel"])}],
                    "severe": [{"name": "Dermatologist Treatment Plan", "category": "medical_treatment",
                        "confidence": round(confidence, 2), "cost": 150.00, "duration": "3-6 months",
                        "steps": (["Dermatologist Consultation",
                            "Prescription Strength Treatments (Isotretinoin, etc.)",
                            "In-Office Procedures (Chemical Peels/Extraction)",
                            "Medicated Topical Treatments", "Follow-up Appointments"]
                            if gender == "female" else
                            ["Dermatologist Consultation", "Prescription Treatments (Isotretinoin)",
                            "In-Office Extractions", "Strong Topical Treatments", "Follow-up Visits"])
                            if sleep_time == "greater_than_6hrs" else
                            (["Dermatologist Consultation", "Milder Prescription Treatments",
                            "Gentle In-Office Procedures", "Soothing Topical Treatments",
                            "Regular Follow-ups"]
                            if gender == "female" else
                            ["Dermatologist Consultation", "Light Prescription Treatments",
                            "Basic Procedures", "Mild Topical Treatments", "Regular Check-ins"])}],
                }
            },
            "young_adult": {
                "early_aging": {
                    "mild": [{"name": "Natural Anti-Aging Home Remedies", "category": "home_remedy",
                        "confidence": round(confidence, 2), "cost": 15.99, "duration": "6 weeks",
                        "steps": (["Avocado and Honey Face Mask (weekly)", "Green Tea Toner",
                            "DIY Vitamin C Serum (Orange/Lemon Extract)", "Coconut Oil Moisturizer"]
                            if gender == "female" else
                            ["Avocado Mask (weekly)", "Green Tea Toner",
                            "Simple Vitamin C Serum", "Light Coconut Moisturizer"])
                            if sleep_time == "greater_than_6hrs" else
                            (["Gentle Oatmeal Mask (weekly)", "Chamomile Toner",
                            "Light Vitamin C Serum", "Shea Butter Light Moisturizer"]
                            if gender == "female" else
                            ["Basic Oatmeal Mask", "Simple Toner",
                            "Mild Vitamin C Serum", "Light Shea Moisturizer"])}],
                    "moderate": [{"name": "Natural Anti-Aging System", "category": "home_remedy",
                        "confidence": round(confidence, 2), "cost": 24.99, "duration": "8 weeks",
                        "steps": (["Oatmeal and Yogurt Cleanser", "Rosehip Oil Serum",
                            "Coffee Ground Exfoliator (2x weekly)", "Cucumber Eye Treatment",
                            "Shea Butter Moisturizer"]
                            if gender == "female" else
                            ["Oatmeal Cleanser", "Rosehip Oil Serum",
                            "Simple Exfoliator (2x weekly)", "Cooling Eye Treatment",
                            "Light Shea Moisturizer"])
                            if sleep_time == "greater_than_6hrs" else
                            (["Milk Cleanser", "Light Rosehip Serum",
                            "Gentle Rice Powder Exfoliator", "Cucumber Soothing Treatment",
                            "Light Shea Moisturizer"]
                            if gender == "female" else
                            ["Basic Milk Cleanser", "Mild Rosehip Serum",
                            "Light Exfoliator", "Simple Cooling Treatment", "Basic Moisturizer"])}],
                    "severe": [{"name": "Dermatologist Anti-Aging Treatment Plan",
                        "category": "medical_treatment",
                        "confidence": round(confidence, 2), "cost": 350.00, "duration": "3-6 months",
                        "steps": (["Dermatologist Consultation",
                            "Professional Strength Retinol Treatment",
                            "Botox/Filler Injections (if recommended)",
                            "Professional Chemical Peels",
                            "Customized Medical-Grade Skincare Regimen", "Follow-up Appointments"]
                            if gender == "female" else
                            ["Dermatologist Consultation", "Strong Retinol Treatment",
                            "Botox Injections", "Chemical Peels",
                            "Medical-Grade Skincare", "Follow-up Visits"])
                            if sleep_time == "greater_than_6hrs" else
                            (["Dermatologist Consultation", "Milder Retinol Treatment",
                            "Light Filler Options", "Gentle Chemical Peels",
                            "Soothing Skincare Regimen", "Regular Check-ins"]
                            if gender == "female" else
                            ["Dermatologist Consultation", "Light Retinol Treatment",
                            "Mild Botox", "Basic Peels", "Simple Skincare",
                            "Regular Follow-ups"])}],
                }
            },
            "adult": {
                "aging": {
                    "mild": [{"name": "DIY Age Delay Remedies", "category": "home_remedy",
                        "confidence": round(confidence, 2), "cost": 18.99, "duration": "8 weeks",
                        "steps": (["Egg White Facial Tightening Mask (weekly)",
                            "Green Tea and Rice Water Toner", "Vitamin E Oil Treatment",
                            "Banana and Papaya Enzyme Mask", "Almond Oil Night Treatment"]
                            if gender == "female" else
                            ["Egg White Mask (weekly)", "Green Tea Toner",
                            "Vitamin E Treatment", "Simple Banana Mask", "Light Almond Oil"])
                            if sleep_time == "greater_than_6hrs" else
                            (["Gentle Egg White Mask (weekly)",
                            "Chamomile and Rice Water Toner", "Light Vitamin E Treatment",
                            "Mild Banana Mask", "Soothing Almond Oil"]
                            if gender == "female" else
                            ["Basic Egg Mask", "Simple Toner", "Mild Vitamin E",
                            "Light Banana Mask", "Basic Almond Oil"])}],
                    "moderate": [{"name": "Natural Age Intervention", "category": "home_remedy",
                        "confidence": round(confidence, 2), "cost": 29.99, "duration": "10 weeks",
                        "steps": (["Yogurt and Lemon Cleanser",
                            "DIY Hyaluronic Acid Serum (Aloe Base)",
                            "Turmeric and Honey Mask (weekly)", "Rosehip Oil Eye Treatment",
                            "Avocado and Cocoa Butter Night Cream"]
                            if gender == "female" else
                            ["Yogurt Cleanser", "Simple Hyaluronic Serum",
                            "Clay Mask (weekly)", "Rosehip Oil Treatment",
                            "Light Cocoa Butter Cream"])
                            if sleep_time == "greater_than_6hrs" else
                            (["Mild Yogurt Cleanser", "Light Aloe Serum",
                            "Gentle Turmeric Mask (weekly)", "Soothing Rosehip Treatment",
                            "Light Cocoa Butter Cream"]
                            if gender == "female" else
                            ["Basic Yogurt Cleanser", "Mild Serum", "Light Clay Mask",
                            "Basic Rosehip Treatment", "Simple Cream"])}],
                    "severe": [{"name": "Comprehensive Medical Age Reversal Program",
                        "category": "medical_treatment",
                        "confidence": round(confidence, 2), "cost": 450.00, "duration": "4-8 months",
                        "steps": (["Dermatologist/Plastic Surgeon Consultation",
                            "Laser Skin Resurfacing Treatments",
                            "Injectable Treatments (Botox/Fillers)",
                            "Professional Chemical Peels",
                            "Prescription Retinoid Treatment",
                            "Medical-Grade Skincare Products", "Nutritional Supplements",
                            "Regular Follow-up Appointments"]
                            if gender == "female" else
                            ["Dermatologist Consultation", "Laser Treatments",
                            "Botox Injections", "Chemical Peels",
                            "Strong Retinoid Treatment", "Medical Skincare",
                            "Supplements", "Follow-up Visits"])
                            if sleep_time == "greater_than_6hrs" else
                            (["Dermatologist Consultation", "Gentle Laser Treatments",
                            "Light Injectable Options", "Mild Chemical Peels",
                            "Soothing Retinoid Treatment", "Gentle Skincare Products",
                            "Basic Supplements", "Regular Check-ins"]
                            if gender == "female" else
                            ["Dermatologist Consultation", "Mild Laser Treatments",
                            "Light Botox", "Basic Peels", "Mild Retinoid",
                            "Simple Skincare", "Light Supplements", "Regular Follow-ups"])}],
                }
            },
            "senior": {
                "deep_wrinkles": {
                    "mild": [{"name": "Gentle Senior Skin Home Care", "category": "home_remedy",
                        "confidence": round(confidence, 2), "cost": 22.99, "duration": "10 weeks",
                        "steps": (["Rice Water Cleanser", "Olive Oil and Vitamin E Treatment",
                            "Aloe Vera and Honey Mask (weekly)",
                            "Grapeseed Oil Night Treatment", "Coconut Oil Moisturizer"]
                            if gender == "female" else
                            ["Rice Water Cleanser", "Vitamin E Treatment",
                            "Simple Aloe Mask (weekly)", "Light Grapeseed Oil",
                            "Basic Coconut Moisturizer"])
                            if sleep_time == "greater_than_6hrs" else
                            (["Gentle Rice Water Cleanser", "Light Olive Oil Treatment",
                            "Soothing Aloe Mask (weekly)", "Mild Grapeseed Oil",
                            "Light Coconut Moisturizer"]
                            if gender == "female" else
                            ["Basic Rice Cleanser", "Mild Vitamin E",
                            "Light Aloe Mask", "Simple Grapeseed Oil", "Basic Moisturizer"])}],
                    "moderate": [{"name": "Advanced Natural Wrinkle Care", "category": "home_remedy",
                        "confidence": round(confidence, 2), "cost": 32.99, "duration": "12 weeks",
                        "steps": (["Milk and Oatmeal Cleanser", "Cucumber and Aloe Toner",
                            "DIY Peptide Serum (Rice Water Base)",
                            "Avocado and Honey Mask (weekly)",
                            "Shea Butter and Essential Oil Night Cream",
                            "Facial Massage Techniques"]
                            if gender == "female" else
                            ["Milk Cleanser", "Cucumber Toner", "Simple Peptide Serum",
                            "Avocado Mask (weekly)", "Light Shea Cream", "Basic Massage"])
                            if sleep_time == "greater_than_6hrs" else
                            (["Light Milk Cleanser", "Soothing Cucumber Toner",
                            "Gentle Peptide Serum", "Mild Avocado Mask (weekly)",
                            "Light Shea Cream", "Gentle Massage"]
                            if gender == "female" else
                            ["Basic Milk Cleanser", "Light Toner", "Mild Serum",
                            "Simple Avocado Mask", "Basic Cream", "Light Massage"])}],
                    "severe": [{"name": "Senior Dermatology Treatment Program",
                        "category": "medical_treatment",
                        "confidence": round(confidence, 2), "cost": 550.00, "duration": "4-10 months",
                        "steps": (["Geriatric Dermatologist Consultation",
                            "Radio Frequency Skin Tightening Treatment",
                            "Fractional Laser Therapy", "Deep Dermal Fillers",
                            "Professional Microneedling Sessions",
                            "Medical-Grade Skincare Regimen",
                            "Monthly Follow-up Appointments",
                            "Comprehensive Skin Health Monitoring"]
                            if gender == "female" else
                            ["Dermatologist Consultation", "RF Tightening Treatment",
                            "Laser Therapy", "Fillers", "Microneedling",
                            "Medical Skincare", "Follow-up Visits", "Basic Monitoring"])
                            if sleep_time == "greater_than_6hrs" else
                            (["Geriatric Dermatologist Consultation", "Gentle RF Treatment",
                            "Mild Laser Therapy", "Light Fillers", "Gentle Microneedling",
                            "Soothing Skincare Regimen", "Regular Check-ins", "Basic Monitoring"]
                            if gender == "female" else
                            ["Dermatologist Consultation", "Light RF Treatment",
                            "Basic Laser", "Mild Fillers", "Light Microneedling",
                            "Simple Skincare", "Regular Follow-ups", "Light Monitoring"])}],
                }
            },
        }

        age_category = user_data.get("age_category", "young_adult")
        concern = user_data.get("concern_type", "early_aging")
        severity = user_data.get("severity", "mild")

        age_data = severity_based_recommendations.get(
            age_category, severity_based_recommendations["young_adult"])
        concern_data = age_data.get(concern, next(iter(age_data.values())))
        recommendations = concern_data.get(severity, [])

        if not recommendations:
            recommendations = next(iter(concern_data.values()), [])

        mapping = {"home": "home_remedy", "mixed": "home_remedy", "medical": "medical_treatment"}
        target_category = mapping.get(treatment_type, "home_remedy")
        filtered = [r for r in recommendations if r["category"] == target_category]
        return filtered if filtered else recommendations

    def format_hair_recommendations(self, probs, user_data, treatment_type, confidence):
        stress_level = user_data.get("stress_level", "moderate")
        gender = user_data.get("gender", "female")

        recs = {
            "teen": {
                "dandruff": {
                    "low": [{"name": "Natural Dandruff Home Remedy Kit", "category": "home_remedy",
                        "confidence": round(confidence, 2), "cost": 14.99, "duration": "4 weeks",
                        "steps": (["Apple Cider Vinegar Rinse (2x weekly)",
                            "Tea Tree Oil Scalp Treatment",
                            "Coconut Oil and Lemon Scalp Massage", "Aloe Vera Gel Scalp Application"]
                            if gender == "female" else
                            ["Vinegar Rinse (2x weekly)", "Tea Tree Oil Treatment",
                            "Coconut Oil Massage", "Light Aloe Application"])
                            if stress_level in ["low", "moderate"] else
                            (["Gentle Vinegar Rinse", "Light Tea Tree Treatment",
                            "Soothing Coconut Oil Massage", "Aloe Vera Calming Gel"]
                            if gender == "female" else
                            ["Basic Vinegar Rinse", "Mild Tea Tree",
                            "Light Coconut Massage", "Simple Aloe Gel"])}],
                    "moderate": [{"name": "DIY Dandruff Management System", "category": "home_remedy",
                        "confidence": round(confidence, 2), "cost": 19.99, "duration": "6 weeks",
                        "steps": (["Baking Soda Paste Scalp Treatment",
                            "Apple Cider Vinegar Rinse", "Tea Tree and Rosemary Oil Blend",
                            "Yogurt and Honey Scalp Mask (weekly)", "Aloe Vera Gel Application"]
                            if gender == "female" else
                            ["Baking Soda Treatment", "Vinegar Rinse",
                            "Tea Tree Oil Blend", "Clay Mask (weekly)", "Light Aloe Gel"])
                            if stress_level in ["low", "moderate"] else
                            (["Mild Baking Soda Treatment", "Light Vinegar Rinse",
                            "Gentle Essential Oil Blend", "Oatmeal Scalp Mask",
                            "Light Aloe Application"]
                            if gender == "female" else
                            ["Basic Baking Soda", "Mild Vinegar Rinse",
                            "Light Oil Blend", "Simple Oatmeal Mask", "Basic Aloe"])}],
                    "high": [{"name": "Dermatologist Scalp Treatment Plan",
                        "category": "medical_treatment",
                        "confidence": round(confidence, 2), "cost": 180.00, "duration": "2-4 months",
                        "steps": (["Dermatologist Consultation",
                            "Prescription Anti-Fungal Treatments",
                            "Professional Scalp Analysis", "Medicated Shampoo Prescription",
                            "Steroid Treatment (if needed)", "Follow-up Appointments",
                            "Lifestyle and Diet Recommendations"]
                            if gender == "female" else
                            ["Dermatologist Consultation", "Anti-Fungal Treatments",
                            "Scalp Analysis", "Medicated Shampoo",
                            "Steroid Option", "Follow-up Visits", "Diet Advice"])
                            if stress_level in ["low", "moderate"] else
                            (["Dermatologist Consultation", "Milder Anti-Fungal Treatments",
                            "Basic Scalp Analysis", "Gentle Medicated Shampoo",
                            "Light Steroid Option", "Regular Check-ins", "Stress Management Plan"]
                            if gender == "female" else
                            ["Dermatologist Consultation", "Light Anti-Fungal",
                            "Simple Analysis", "Mild Shampoo",
                            "Basic Steroid", "Regular Follow-ups", "Stress Plan"])}],
                },
                "frizz": {
                    "low": [{"name": "Natural Frizz Control Kit", "category": "home_remedy",
                        "confidence": round(confidence, 2), "cost": 15.99, "duration": "5 weeks",
                        "steps": (["Argan Oil Hair Serum", "Coconut Milk Deep Conditioning",
                            "Silk Pillowcase Usage", "Microfiber Towel Drying"]
                            if gender == "female" else
                            ["Argan Oil Serum", "Coconut Conditioning",
                            "Gentle Drying", "Basic Hair Care"])
                            if stress_level in ["low", "moderate"] else
                            (["Light Argan Oil", "Gentle Coconut Treatment",
                            "Stress-Relief Hair Care", "Soothing Routine"]
                            if gender == "female" else
                            ["Basic Argan Oil", "Simple Conditioning",
                            "Light Hair Care", "Basic Routine"])}],
                    "high": [{"name": "Professional Frizz Treatment",
                        "category": "medical_treatment",
                        "confidence": round(confidence, 2), "cost": 250.00, "duration": "4-6 months",
                        "steps": (["Professional Hair Analysis", "Keratin Treatment Application",
                            "Brazilian Blowout (if suitable)",
                            "Professional-Grade Products", "Regular Maintenance Sessions"]
                            if gender == "female" else
                            ["Hair Analysis", "Keratin Treatment",
                            "Professional Products", "Maintenance"])
                            if stress_level in ["low", "moderate"] else
                            (["Gentle Hair Analysis", "Mild Keratin Treatment",
                            "Light Professional Products", "Stress-Adapted Maintenance"]
                            if gender == "female" else
                            ["Basic Analysis", "Light Keratin",
                            "Simple Products", "Basic Maintenance"])}],
                },
            },
            "young_adult": {
                "breakage": {
                    "low": [{"name": "Natural Hair Strengthening Kit", "category": "home_remedy",
                        "confidence": round(confidence, 2), "cost": 16.99, "duration": "6 weeks",
                        "steps": (["Egg and Olive Oil Hair Mask (weekly)",
                            "Coconut Oil Deep Conditioning",
                            "Aloe Vera Leave-in Treatment", "Protein-Rich Diet Support"]
                            if gender == "female" else
                            ["Egg and Oil Mask (weekly)", "Coconut Conditioning",
                            "Aloe Treatment", "Protein Diet"])
                            if stress_level in ["low", "moderate"] else
                            (["Gentle Egg Mask", "Light Coconut Oil",
                            "Soothing Aloe", "Balanced Diet"]
                            if gender == "female" else
                            ["Basic Egg Mask", "Simple Coconut Oil",
                            "Light Aloe", "Basic Diet"])}],
                    "high": [{"name": "Professional Hair Repair Treatment",
                        "category": "medical_treatment",
                        "confidence": round(confidence, 2), "cost": 280.00, "duration": "3-6 months",
                        "steps": (["Trichologist Consultation",
                            "Professional Bond-Building Treatments",
                            "Olaplex or Similar Treatments",
                            "Medical-Grade Hair Care Products",
                            "Nutritional Supplements", "Regular Progress Monitoring"]
                            if gender == "female" else
                            ["Trichologist Consultation", "Bond-Building Treatments",
                            "Professional Products", "Supplements", "Monitoring"])
                            if stress_level in ["low", "moderate"] else
                            (["Trichologist Consultation", "Gentle Bond-Building",
                            "Light Professional Products", "Stress-Adapted Supplements",
                            "Regular Check-ins"]
                            if gender == "female" else
                            ["Consultation", "Light Treatments",
                            "Basic Products", "Simple Supplements", "Basic Monitoring"])}],
                },
                "early_hair_loss": {
                    "moderate": [{"name": "Enhanced Growth Protocol", "category": "home_remedy",
                        "confidence": round(confidence, 2), "cost": 29.99, "duration": "10 weeks",
                        "steps": (["Castor Oil and Peppermint Scalp Treatment",
                            "Fenugreek Seed Hair Mask (weekly)",
                            "Scalp Stimulation Massage",
                            "Biotin and Zinc Supplements", "Stress Management Techniques"]
                            if gender == "female" else
                            ["Castor Oil Treatment", "Fenugreek Mask (weekly)",
                            "Scalp Massage", "Supplements", "Stress Management"])
                            if stress_level in ["low", "moderate"] else
                            (["Gentle Castor Oil Treatment", "Light Fenugreek Mask",
                            "Soothing Massage", "Stress-Adapted Supplements",
                            "Relaxation Techniques"]
                            if gender == "female" else
                            ["Basic Castor Oil", "Simple Fenugreek",
                            "Light Massage", "Basic Supplements", "Simple Relaxation"])}],
                    "high": [{"name": "Medical Hair Loss Treatment",
                        "category": "medical_treatment",
                        "confidence": round(confidence, 2), "cost": 400.00, "duration": "6-12 months",
                        "steps": (["Dermatologist/Trichologist Consultation",
                            "Minoxidil Treatment Prescription",
                            "PRP (Platelet-Rich Plasma) Therapy",
                            "Professional Scalp Treatments",
                            "Hormonal Testing and Balance",
                            "Medical-Grade Supplements", "Regular Progress Monitoring"]
                            if gender == "female" else
                            ["Dermatologist Consultation", "Minoxidil Prescription",
                            "PRP Therapy", "Scalp Treatments",
                            "Testing", "Supplements", "Monitoring"])
                            if stress_level in ["low", "moderate"] else
                            (["Consultation with Stress Focus", "Gentle Minoxidil Treatment",
                            "Modified PRP Therapy", "Stress-Adapted Scalp Treatments",
                            "Comprehensive Testing", "Stress-Support Supplements",
                            "Frequent Check-ins"]
                            if gender == "female" else
                            ["Basic Consultation", "Light Minoxidil",
                            "Basic PRP", "Simple Treatments",
                            "Basic Testing", "Simple Supplements", "Regular Monitoring"])}],
                },
            },
            "adult": {
                "thinning": {
                    "low": [{"name": "Natural Hair Density Support", "category": "home_remedy",
                        "confidence": round(confidence, 2), "cost": 21.99, "duration": "10 weeks",
                        "steps": (["Saw Palmetto Hair Rinse",
                            "Rosemary and Lavender Oil Treatment",
                            "Protein-Rich Hair Masks", "Scalp Microcirculation Massage",
                            "Nutritional Support"]
                            if gender == "female" else
                            ["Saw Palmetto Rinse", "Rosemary Oil Treatment",
                            "Protein Masks", "Scalp Massage", "Nutrition"])
                            if stress_level in ["low", "moderate"] else
                            (["Gentle Saw Palmetto", "Light Oil Treatment",
                            "Soothing Protein Masks", "Relaxing Massage",
                            "Stress-Reducing Nutrition"]
                            if gender == "female" else
                            ["Basic Saw Palmetto", "Simple Oil Treatment",
                            "Light Protein Masks", "Basic Massage", "Simple Nutrition"])}],
                    "high": [{"name": "Comprehensive Hair Restoration Program",
                        "category": "medical_treatment",
                        "confidence": round(confidence, 2), "cost": 500.00, "duration": "6-18 months",
                        "steps": (["Comprehensive Hair Loss Assessment",
                            "Prescription Treatments (Finasteride/Minoxidil)",
                            "PRP Therapy Sessions", "Low-Level Laser Therapy",
                            "Hormonal Balance Treatment", "Medical-Grade Supplements",
                            "Possible Hair Transplant Consultation",
                            "Regular Monitoring and Adjustment"]
                            if gender == "female" else
                            ["Hair Loss Assessment", "Prescription Treatments",
                            "PRP Therapy", "Laser Therapy", "Hormonal Treatment",
                            "Supplements", "Transplant Consultation", "Monitoring"])
                            if stress_level in ["low", "moderate"] else
                            (["Stress-Focused Assessment", "Gentle Prescription Treatments",
                            "Modified PRP Therapy", "Relaxing Laser Therapy",
                            "Stress-Adapted Hormonal Treatment",
                            "Stress-Support Supplements",
                            "Conservative Transplant Discussion",
                            "Frequent Progress Reviews"]
                            if gender == "female" else
                            ["Basic Assessment", "Light Prescriptions",
                            "Basic PRP", "Simple Laser", "Basic Hormonal",
                            "Simple Supplements", "Basic Consultation", "Regular Reviews"])}],
                },
            },
            "senior": {
                "significant_thinning": {
                    "low": [{"name": "Gentle Hair Care for Mature Hair", "category": "home_remedy",
                        "confidence": round(confidence, 2), "cost": 24.99, "duration": "10 weeks",
                        "steps": (["Gentle Argan Oil Treatment", "Biotin Supplements",
                            "Scalp Nourishment Massage", "Silk Pillowcase Usage"]
                            if gender == "female" else
                            ["Argan Oil Treatment", "Biotin",
                            "Scalp Massage", "Silk Pillowcase"])
                            if stress_level in ["low", "moderate"] else
                            (["Very Gentle Argan Treatment", "Stress-Adapted Biotin",
                            "Soothing Massage", "Comfortable Sleeping"]
                            if gender == "female" else
                            ["Light Argan Oil", "Basic Biotin",
                            "Light Massage", "Basic Pillowcase"])}],
                    "high": [{"name": "Geriatric Hair Restoration Program",
                        "category": "medical_treatment",
                        "confidence": round(confidence, 2), "cost": 450.00, "duration": "8-16 months",
                        "steps": (["Geriatric Trichologist Consultation",
                            "Age-Appropriate Minoxidil Treatment",
                            "Gentle PRP Therapy (if suitable)",
                            "Low-Level Laser Therapy",
                            "Nutritional Deficiency Testing",
                            "Customized Supplement Protocol",
                            "Gentle Scalp Treatments", "Regular Health Monitoring"]
                            if gender == "female" else
                            ["Trichologist Consultation", "Minoxidil Treatment",
                            "PRP Therapy", "Laser Therapy", "Testing",
                            "Supplements", "Scalp Treatments", "Monitoring"])
                            if stress_level in ["low", "moderate"] else
                            (["Comprehensive Geriatric Consultation",
                            "Stress-Adapted Minoxidil", "Modified PRP Therapy",
                            "Relaxing Laser Therapy", "Full Health Assessment",
                            "Stress-Support Supplement Plan",
                            "Soothing Scalp Treatments", "Frequent Health Check-ins"]
                            if gender == "female" else
                            ["Basic Consultation", "Light Minoxidil",
                            "Gentle PRP", "Simple Laser", "Basic Testing",
                            "Simple Supplements", "Basic Scalp Care", "Regular Monitoring"])}],
                },
                "gray_hair": {
                    "low": [{"name": "Natural Gray Hair Enhancement", "category": "home_remedy",
                        "confidence": round(confidence, 2), "cost": 16.99, "duration": "6 weeks",
                        "steps": (["Purple Shampoo for Brightness", "Argan Oil Shine Treatment",
                            "Gentle Clarifying Rinse", "Moisturizing Hair Masks"]
                            if gender == "female" else
                            ["Purple Shampoo", "Argan Oil",
                            "Clarifying Rinse", "Moisturizing Masks"])
                            if stress_level in ["low", "moderate"] else
                            (["Gentle Purple Shampoo", "Light Argan Treatment",
                            "Mild Clarifying Rinse", "Soothing Hair Masks"]
                            if gender == "female" else
                            ["Basic Purple Shampoo", "Simple Argan",
                            "Light Rinse", "Basic Masks"])}],
                    "high": [{"name": "Professional Silver Hair Management",
                        "category": "medical_treatment",
                        "confidence": round(confidence, 2), "cost": 200.00, "duration": "Ongoing",
                        "steps": (["Professional Gray Hair Consultation",
                            "Salon-Grade Toning Treatments",
                            "Professional Deep Conditioning",
                            "Customized Product Regimen", "Regular Maintenance Appointments"]
                            if gender == "female" else
                            ["Hair Consultation", "Toning Treatments",
                            "Deep Conditioning", "Product Regimen", "Maintenance"])
                            if stress_level in ["low", "moderate"] else
                            (["Gentle Professional Consultation", "Stress-Adapted Toning",
                            "Soothing Deep Conditioning", "Gentle Product Regimen",
                            "Flexible Appointments"]
                            if gender == "female" else
                            ["Basic Consultation", "Light Toning",
                            "Simple Conditioning", "Basic Products", "Regular Maintenance"])}],
                },
            },
        }

        age_category = user_data.get("age_category", "young_adult")
        concern = user_data.get("hair_concern", "breakage")
        severity = user_data.get("hair_damage_level", "low")

        if concern not in recs.get(age_category, {}):
            concern = HAIR_CONCERN_FALLBACK.get(concern, "breakage")

        age_data = recs.get(age_category, recs["young_adult"])
        concern_data = age_data.get(concern, next(iter(age_data.values()), {}))
        recommendations = concern_data.get(severity, [])

        if not recommendations:
            recommendations = next(iter(concern_data.values()), [])

        mapping = {"home": "home_remedy", "mixed": "home_remedy", "medical": "medical_treatment"}
        target_category = mapping.get(treatment_type, "home_remedy")
        filtered = [r for r in recommendations if r["category"] == target_category]
        return filtered if filtered else recommendations



def create_comprehensive_dataset(n_samples: int = 8000) -> list[dict]:
    np.random.seed(42)
    data = []
    age_ranges = {
        "teen":        list(range(13, 20)),
        "young_adult": list(range(20, 35)),
        "adult":       list(range(35, 50)),
        "senior":      list(range(50, 75)),
    }
    skin_types  = ["oily", "dry", "combination", "sensitive", "normal"]
    severities  = ["mild", "moderate", "severe"]
    damages     = ["low", "moderate", "high"]
    sleep_opts  = ["greater_than_6hrs", "less_than_6hrs"]
    stress_opts = ["low", "moderate", "high"]
    genders     = ["female", "male"]

    for _ in range(n_samples):
        age_cat = np.random.choice(list(age_ranges.keys()))
        age     = int(np.random.choice(age_ranges[age_cat]))
        sleep   = np.random.choice(sleep_opts, p=[0.65, 0.35])
        stress  = np.random.choice(stress_opts, p=[0.25, 0.45, 0.30])

        if age_cat == "teen":
            concern = np.random.choice(
                ["acne", "oiliness", "sensitivity"],
                p=[0.70, 0.15, 0.15] if sleep == "less_than_6hrs"
                else [0.55, 0.20, 0.25]
            )
        elif age_cat == "young_adult":
            concern = np.random.choice(
                ["early_aging", "hyperpigmentation", "uneven_texture"],
                p=[0.65, 0.20, 0.15]
            )
        elif age_cat == "adult":
            concern = np.random.choice(
                ["aging", "dark_spots", "fine_lines"],
                p=[0.60, 0.25, 0.15]
            )
        else:
            concern = np.random.choice(
                ["deep_wrinkles", "dryness", "age_spots"],
                p=[0.70, 0.15, 0.15]
            )

        if stress == "high" and age_cat in ["adult", "senior"]:
            hair = np.random.choice(
                ["thinning", "significant_thinning", "damage_from_styling"],
                p=[0.50, 0.35, 0.15]
            )
        elif stress == "high":
            hair = np.random.choice(
                ["breakage", "frizz", "early_hair_loss"],
                p=[0.50, 0.30, 0.20]
            )
        elif age_cat == "senior":
            hair = np.random.choice(
                ["significant_thinning", "gray_hair", "scalp_health"],
                p=[0.50, 0.30, 0.20]
            )
        elif age_cat == "adult":
            hair = np.random.choice(
                ["thinning", "gray_coverage", "damage_from_styling"],
                p=[0.45, 0.30, 0.25]
            )
        else:
            hair = np.random.choice(
                ["breakage", "frizz", "dandruff"],
                p=[0.40, 0.35, 0.25]
            )

        base_sev  = np.random.choice([0, 1, 2], p=[0.40, 0.40, 0.20])
        noise_sev = np.clip(base_sev + np.random.choice([-1, 0, 1], p=[0.1, 0.8, 0.1]), 0, 2)
        severity  = severities[int(noise_sev)]

        base_dmg = np.random.choice([0, 1, 2], p=[0.45, 0.35, 0.20])
        damage   = damages[int(base_dmg)]

        entry = {
            "age":               age,
            "age_category":      age_cat,
            "skin_type":         np.random.choice(skin_types),
            "concern_type":      concern,
            "severity":          severity,
            "sleep_time":        sleep,
            "stress_level":      stress,
            "gender":            np.random.choice(genders),
            "hair_concern":      hair,
            "hair_damage_level": damage,
        }
        data.append(entry)

        raw_score = score_treatment(pd.Series(entry))
        lo_h, hi_h = BOUNDARY_ZONES["home_mixed"]
        lo_m, hi_m = BOUNDARY_ZONES["mixed_medical"]
        is_boundary = (lo_h <= raw_score <= hi_h) or (lo_m <= raw_score <= hi_m)
        if is_boundary and np.random.random() < 0.60:
            data.append(entry.copy())


        if raw_score >= 8 and np.random.random() < 0.50:
            data.append(entry.copy())

    return data




def get_user_input(recommendation_type: str) -> dict:
    user_data = {}
    print(f"\n=== {recommendation_type.upper()} CARE RECOMMENDATION SYSTEM ===")

    while True:
        try:
            age = int(input("Enter your age: "))
            if age < 13:
                print("Sorry, this system is for ages 13+.")
                continue
            user_data["age"] = age
            user_data["age_category"] = categorize_age(age)
            break
        except ValueError:
            print("Please enter a valid number.")

    print("\nGender options: female, male")
    user_data["gender"] = validate(input("Gender: "), ["female", "male"], "female")

    if recommendation_type == "skin":
        print("\nSleep options: greater_than_6hrs, less_than_6hrs")
        user_data["sleep_time"] = validate(
            input("Sleep per night: "),
            ["greater_than_6hrs", "less_than_6hrs"], "greater_than_6hrs")
        user_data["stress_level"] = "moderate"

        skin_allowed = {
            "teen":        ["acne", "oiliness", "sensitivity"],
            "young_adult": ["early_aging", "hyperpigmentation", "uneven_texture"],
            "adult":       ["aging", "fine_lines", "dark_spots"],
            "senior":      ["deep_wrinkles", "age_spots", "dryness"],
        }
        print(f"\nSkin type options: oily, dry, combination, sensitive, normal")
        user_data["skin_type"] = validate(
            input("Skin type: "),
            ["oily", "dry", "combination", "sensitive", "normal"], "normal")

        print(f"\nSkin concern options: {', '.join(skin_allowed[user_data['age_category']])}")
        user_data["concern_type"] = validate(
            input("Primary skin concern: "),
            skin_allowed[user_data["age_category"]],
            skin_allowed[user_data["age_category"]][0])

        print("\nSeverity options: mild, moderate, severe")
        user_data["severity"] = validate(
            input("Severity: "),
            ["mild", "moderate", "severe"], "mild")

        user_data["hair_concern"]      = "breakage"
        user_data["hair_damage_level"] = "low"

    elif recommendation_type == "hair":
        print("\nStress level options: low, moderate, high")
        user_data["stress_level"] = validate(
            input("Stress level: "),
            ["low", "moderate", "high"], "moderate")
        user_data["sleep_time"] = "greater_than_6hrs"

        print("\nHair damage options: low, moderate, high")
        user_data["hair_damage_level"] = validate(
            input("Hair damage level: "),
            ["low", "moderate", "high"], "low")

        hair_allowed = {
            "teen":        ["dandruff", "oily_scalp", "frizz"],
            "young_adult": ["breakage", "frizz", "early_hair_loss"],
            "adult":       ["thinning", "gray_coverage", "damage_from_styling"],
            "senior":      ["significant_thinning", "gray_hair", "scalp_health"],
        }
        print(f"\nHair concern options: {', '.join(hair_allowed[user_data['age_category']])}")
        user_data["hair_concern"] = validate(
            input("Primary hair concern: "),
            hair_allowed[user_data["age_category"]],
            hair_allowed[user_data["age_category"]][0])

        user_data["concern_type"] = "aging"
        user_data["severity"]     = "mild"
        user_data["skin_type"]    = "normal"

    return user_data




MODEL_PATH = "selfcare_model_v3.pkl"


def main():
    recommender = None

    if os.path.exists(MODEL_PATH):
        print("Loading saved model...")
        try:
            recommender = joblib.load(MODEL_PATH)
            required = ["label_encoders", "feature_columns", "calibrated_model", "cf"]
            missing  = [a for a in required if not hasattr(recommender, a)
                        or getattr(recommender, a) is None]
            if missing:
                print(f"Model incomplete ({missing}). Retraining...")
                recommender = None
            else:
                print("Model loaded successfully!")
        except Exception as e:
            print(f"Error loading: {e}. Retraining...")
            recommender = None

    if recommender is None:
        print("\nTraining enhanced ML model (v3 — all fixes applied)...")
        recommender = SelfCareRecommender(n_neighbors=10, cf_weight=0.25)

        raw_data = create_comprehensive_dataset(n_samples=8000)
        df = pd.DataFrame(raw_data)
        y  = df.apply(lambda row: assign_treatment(row, add_noise=True), axis=1)

        X_processed = recommender.fit_preprocess(df)
        metrics     = recommender.train_model(X_processed, y)

        recommender.print_feature_importances(top_n=12)

        print("\nSaving model...")
        joblib.dump(recommender, MODEL_PATH)
        print(f"Model saved → {MODEL_PATH}")

    # ── Main loop ──
    while True:
        print("\n" + "═"*50)
        print("  SELF-CARE RECOMMENDATION SYSTEM  (ML v3)")
        print("═"*50)
        print("  1. Skin Care Recommendations")
        print("  2. Hair Care Recommendations")
        print("  3. Exit")
        choice = input("Choice (1-3): ").strip()

        if choice == "1":
            user_data = get_user_input("skin")
            result    = recommender.get_recommendations(user_data)

            print("\n" + "─"*50)
            print("  PERSONALIZED SKIN CARE RECOMMENDATIONS")
            print("─"*50)

            exp = result.get("explanation", {})
            tier = result.get("uncertainty_tier", "")
            tier_label = {
                "high_confidence": "✓ High confidence",
                "moderate_confidence": "~ Moderate confidence",
                "low_confidence_consider_consultation": "⚠ Low confidence — consider professional consultation"
            }.get(tier, tier)
            print(f"\n  Treatment Type : {exp.get('predicted_treatment','').upper()}")
            print(f"  Confidence     : {exp.get('confidence_pct','')}  [{tier_label}]")
            print(f"  Key Drivers    : {', '.join(exp.get('top_drivers', []))}")
            print(f"\n  Probability Breakdown:")
            for cls, p in result["model_probabilities"].items():
                bar = "█" * int(p * 30)
                print(f"    {cls:<10} {p:.3f}  {bar}")


            print(f"\n  ⚠  {result['disclaimer']}")

            for rec in result["skin_care"]:
                print(f"\n  ► {rec['name']}")
                print(f"    Category : {rec['category']}")
                print(f"    Cost     : ${rec['cost']}")
                print(f"    Duration : {rec['duration']}")
                print("    Steps:")
                for i, s in enumerate(rec["steps"], 1):
                    print(f"      {i}. {s}")

        elif choice == "2":
            user_data = get_user_input("hair")
            result    = recommender.get_recommendations(user_data)

            print("\n" + "─"*50)
            print("  PERSONALIZED HAIR CARE RECOMMENDATIONS")
            print("─"*50)

            exp = result.get("explanation", {})
            tier = result.get("uncertainty_tier", "")
            tier_label = {
                "high_confidence": "✓ High confidence",
                "moderate_confidence": "~ Moderate confidence",
                "low_confidence_consider_consultation": "⚠ Low confidence — consider professional consultation"
            }.get(tier, tier)
            print(f"\n  Treatment Type : {exp.get('predicted_treatment','').upper()}")
            print(f"  Confidence     : {exp.get('confidence_pct','')}  [{tier_label}]")
            print(f"  Key Drivers    : {', '.join(exp.get('top_drivers', []))}")
            print(f"\n  Probability Breakdown:")
            for cls, p in result["model_probabilities"].items():
                bar = "█" * int(p * 30)
                print(f"    {cls:<10} {p:.3f}  {bar}")


            print(f"\n  ⚠  {result['disclaimer']}")

            for rec in result["hair_care"]:
                print(f"\n  ► {rec['name']}")
                print(f"    Category : {rec['category']}")
                print(f"    Cost     : ${rec['cost']}")
                print(f"    Duration : {rec['duration']}")
                print("    Steps:")
                for i, s in enumerate(rec["steps"], 1):
                    print(f"      {i}. {s}")

        elif choice == "3":
            print("Goodbye!")
            break
        else:
            print("Invalid choice.")


if __name__ == "__main__":
    main()

Loading saved model...
Model loaded successfully!

══════════════════════════════════════════════════
  SELF-CARE RECOMMENDATION SYSTEM  (ML v3)
══════════════════════════════════════════════════
  1. Skin Care Recommendations
  2. Hair Care Recommendations
  3. Exit
Choice (1-3): 1

=== SKIN CARE RECOMMENDATION SYSTEM ===
Enter your age: 26

Gender options: female, male
Gender: male

Sleep options: greater_than_6hrs, less_than_6hrs
Sleep per night: less_than_6hrs

Skin type options: oily, dry, combination, sensitive, normal
Skin type: oily

Skin concern options: early_aging, hyperpigmentation, uneven_texture
Primary skin concern: uneven_texture

Severity options: mild, moderate, severe
Severity: moderate

──────────────────────────────────────────────────
  PERSONALIZED SKIN CARE RECOMMENDATIONS
──────────────────────────────────────────────────

  Treatment Type : HOME
  Confidence     : 53.5%  [⚠ Low confidence — consider professional consultation]
  Key Drivers    : concern_type, h